In [ ]:
#@title 🎧 Download Narration Audio & Play Introduction
import os as _os
if not _os.path.exists("/content/narration"):
    !pip install -q gdown
    import gdown
    gdown.download(id="1sIIbqOhGig9Ht8-9s4r2lsO-Yd4Bu8QC", output="/content/narration.zip", quiet=False)
    !unzip -q /content/narration.zip -d /content/narration
    !rm /content/narration.zip
    print(f"Loaded {len(_os.listdir('/content/narration'))} narration segments")
else:
    print("Narration audio already loaded.")

from IPython.display import Audio, display
display(Audio("/content/narration/01_00_intro.mp3"))

In [ ]:
#@title 🎧 Listen: Setup Code
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_01_setup_code.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

In [ ]:
# 🔧 Setup: Run this cell first!
# Check GPU availability and install dependencies

import torch
import sys

# Check GPU
if torch.cuda.is_available():
    device = torch.device('cuda')
    print(f"✅ GPU available: {torch.cuda.get_device_name(0)}")
    print(f"   Memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")
else:
    device = torch.device('cpu')
    print("⚠️ No GPU detected. Some cells may run slowly.")
    print("   Go to Runtime → Change runtime type → GPU")

print(f"\n📦 Python {sys.version.split()[0]}")
print(f"🔥 PyTorch {torch.__version__}")

# Set random seeds for reproducibility
import random
import numpy as np

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print(f"🎲 Random seed set to {SEED}")

%matplotlib inline

# Token Budgeting & Prompt Compression — Vizuara

# Token Budgeting & Prompt Compression
## Making Every Token Count in Your Context Window

Welcome to the first notebook in the Context Optimization & Evaluation series! Here we will build practical tools for managing your context window's finite token budget and compressing documents to fit more signal into fewer tokens.

**What you will learn:**
- How to allocate tokens across competing context components
- Three compression strategies: extractive, abstractive, and token-level
- How to measure compression quality

**Prerequisites:** Basic Python, familiarity with LLMs and context windows

In [ ]:
#@title 🎧 Listen: Why Matter
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_02_why_matter.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

## 1. Why Does This Matter?

Every LLM has a finite context window — 128K tokens for GPT-4, 200K for Claude. That sounds generous, but in a production system you are splitting that budget across system prompts, conversation history, retrieved documents, tool outputs, and the user's query. If you waste tokens on redundant or irrelevant content, you either exceed the window or crowd out important information.

Think of it like packing a suitcase with a strict weight limit. Every unnecessary item means something essential might be left behind.

Let us start by understanding exactly how much a token costs in terms of information.

In [ ]:
#@title 🎧 Listen: Install Deps
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_03_install_deps.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

In [ ]:
# Install dependencies
!pip install -q tiktoken scikit-learn numpy matplotlib

In [ ]:
#@title 🎧 Listen: Token Density
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_04_token_density.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

In [ ]:
import tiktoken
import numpy as np

# Initialize the tokenizer for GPT-4
enc = tiktoken.encoding_for_model("gpt-4")

# Example texts of varying information density
texts = {
    "Dense (technical)": "BERT uses bidirectional self-attention with 12 transformer layers, 768 hidden units, and 110M parameters, trained on BooksCorpus and English Wikipedia.",
    "Sparse (filler)": "In this particular section of the document, we would like to take the opportunity to discuss and elaborate upon some of the various aspects and considerations.",
    "Medium (explanation)": "The model processes input tokens in parallel through self-attention, allowing each token to attend to every other token in the sequence."
}

print("=== Token Density Analysis ===\n")
for label, text in texts.items():
    tokens = enc.encode(text)
    words = text.split()
    chars = len(text)
    print(f"{label}:")
    print(f"  Words: {len(words)}, Tokens: {len(tokens)}, Chars: {chars}")
    print(f"  Tokens per word: {len(tokens)/len(words):.2f}")
    print(f"  Information density: {'HIGH' if len(tokens)/len(words) < 1.5 else 'LOW'}")
    print()

In [ ]:
#@title 🎧 Listen: Budget Intuition
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_05_budget_intuition.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

## 2. Building Intuition: The Token Budget Constraint

The **budget constraint** is:

$$T_{\text{total}} = T_{\text{system}} + T_{\text{history}} + T_{\text{RAG}} + T_{\text{tools}} + T_{\text{user}} + T_{\text{reserved}}$$

Let us build a simple budget allocator and see what happens when we exceed the limit.

In [ ]:
#@title 🎧 Listen: Budget Class Normal
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_06_budget_class_normal.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

In [ ]:
class TokenBudget:
    """Simple token budget manager for context windows."""

    def __init__(self, window_size: int, reserved_output: int):
        self.window_size = window_size
        self.reserved = reserved_output
        self.available = window_size - reserved_output
        self.allocations = {}

    def allocate(self, component: str, tokens: int):
        self.allocations[component] = tokens

    def get_total_used(self) -> int:
        return sum(self.allocations.values())

    def get_remaining(self) -> int:
        return self.available - self.get_total_used()

    def report(self):
        total = self.get_total_used()
        print(f"\n{'='*50}")
        print(f"Context Window: {self.window_size:,} tokens")
        print(f"Reserved for output: {self.reserved:,} tokens")
        print(f"Available for context: {self.available:,} tokens")
        print(f"{'='*50}")
        for comp, tokens in self.allocations.items():
            pct = tokens / self.available * 100
            bar = '#' * int(pct / 2)
            print(f"  {comp:<25} {tokens:>8,} tokens ({pct:5.1f}%) |{bar}")
        print(f"{'='*50}")
        print(f"  {'TOTAL USED':<25} {total:>8,} tokens")
        remaining = self.get_remaining()
        status = "OK" if remaining >= 0 else "OVER BUDGET!"
        print(f"  {'REMAINING':<25} {remaining:>8,} tokens  [{status}]")
        print(f"{'='*50}")

# Normal allocation
budget = TokenBudget(window_size=128_000, reserved_output=35_000)
budget.allocate("System Prompt", 2_000)
budget.allocate("Conversation History", 15_000)
budget.allocate("RAG Documents", 50_000)
budget.allocate("Tool Results", 10_000)
budget.allocate("User Message", 1_000)
budget.report()

In [ ]:
# What happens when RAG returns too many documents


In [ ]:
#@title 🎧 Listen: Budget Class Over
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_07_budget_class_over.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

In [ ]:
print("\n--- SCENARIO: RAG returns 80K tokens instead of 50K ---")
budget_over = TokenBudget(window_size=128_000, reserved_output=35_000)
budget_over.allocate("System Prompt", 2_000)
budget_over.allocate("Conversation History", 15_000)
budget_over.allocate("RAG Documents", 80_000)
budget_over.allocate("Tool Results", 10_000)
budget_over.allocate("User Message", 1_000)
budget_over.report()

In [ ]:
#@title 🎧 Listen: Compression Math
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_08_compression_math.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

## 3. The Mathematics of Compression

**Compression Ratio:** $$\text{CR} = 1 - \frac{T_{\text{compressed}}}{T_{\text{original}}}$$

**Information Retention:** $$\text{IR}(D', q) = \frac{\sum_{s \in D'} \text{sim}(s, q)}{\sum_{s \in D} \text{sim}(s, q)}$$

In [ ]:
#@title 🎧 Listen: Metrics Function
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_09_metrics_function.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

In [ ]:
def compute_compression_metrics(original_tokens, compressed_tokens,
                                 original_relevance, compressed_relevance):
    cr = 1 - compressed_tokens / original_tokens
    ir = compressed_relevance / original_relevance if original_relevance > 0 else 0
    print(f"Original tokens:    {original_tokens:,}")
    print(f"Compressed tokens:  {compressed_tokens:,}")
    print(f"Compression ratio:  {cr:.1%}")
    print(f"Information retention: {ir:.1%}")
    print(f"Efficiency score:   {cr * ir:.3f} (higher is better)")
    return cr, ir

print("=== Numerical Example ===")
compute_compression_metrics(4000, 800, 15.2, 13.7)

In [ ]:
#@title 🎧 Listen: Extractive Intro
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_10_extractive_intro.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

## 4. Let's Build It — Extractive Compression with TF-IDF

In [ ]:
#@title 🎧 Listen: Extractive Impl
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_11_extractive_impl.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

def extractive_compress(document, query, keep_ratio=0.3):
    sentences = [s.strip() for s in document.split('. ') if len(s.strip()) > 10]
    if len(sentences) <= 3:
        return document

    vectorizer = TfidfVectorizer(stop_words='english')
    all_texts = [query] + sentences
    tfidf_matrix = vectorizer.fit_transform(all_texts)

    query_vec = tfidf_matrix[0:1]
    sentence_vecs = tfidf_matrix[1:]
    scores = cosine_similarity(query_vec, sentence_vecs).flatten()

    k = max(1, int(len(sentences) * keep_ratio))
    top_indices = np.argsort(scores)[-k:]
    top_indices = sorted(top_indices)

    compressed = '. '.join(sentences[i] for i in top_indices) + '.'

    print(f"Sentences: {len(sentences)} -> {k} (kept {keep_ratio:.0%})")
    for i, (score, sent) in enumerate(zip(scores, sentences)):
        marker = ">>>" if i in top_indices else "   "
        print(f"  {marker} [{score:.3f}] {sent[:80]}...")

    return compressed

document = """Retrieval-Augmented Generation combines the power of large language models with external knowledge retrieval. The system first encodes the user query into a dense vector representation using an embedding model. This vector is then used to search a vector database containing pre-indexed document chunks. The vector database returns the top-k most similar chunks based on cosine similarity. These retrieved chunks are concatenated and injected into the LLM's context window alongside the original query. The LLM then generates a response that is grounded in the retrieved information rather than relying solely on its parametric knowledge. This approach significantly reduces hallucination rates compared to vanilla LLM generation. Additionally the system can be enhanced with reranking models that rescore the retrieved chunks for better relevance. Production RAG systems also implement caching strategies to reduce latency for repeated queries. The embedding model and the reranker can be fine-tuned on domain-specific data for improved performance."""

query = "How does RAG reduce hallucination?"
compressed = extractive_compress(document, query, keep_ratio=0.3)

print(f"\n=== Compressed Result ===")
print(compressed)

original_tokens = len(enc.encode(document))
compressed_tokens = len(enc.encode(compressed))
print(f"\nTokens: {original_tokens} -> {compressed_tokens} (CR = {1 - compressed_tokens/original_tokens:.1%})")

In [ ]:
#@title 🎧 Listen: Todo1 Intro
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_12_todo1_intro.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

## 5. Your Turn: Build a Multi-Strategy Compressor

**TODO Exercise 1:** Find the best keep_ratio for extractive compression that fits within a target token budget while maximizing relevance.

In [ ]:
#@title 🎧 Listen: Todo1 Task
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_13_todo1_task.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

In [ ]:
def find_optimal_compression(document, query, target_tokens):
    """
    TODO: Try keep_ratios [0.1, 0.2, 0.3, 0.4, 0.5].
    For each, compute compressed text and its token count.
    Return the compressed text that fits within target_tokens
    and has the highest relevance score.
    """
    best_text = document
    best_score = 0.0

    # YOUR CODE HERE
    # for ratio in [0.1, 0.2, 0.3, 0.4, 0.5]:
    #     compressed = extractive_compress(document, query, keep_ratio=ratio)
    #     tokens = len(enc.encode(compressed))
    #     if tokens <= target_tokens:
    #         vectorizer = TfidfVectorizer(stop_words='english')
    #         tfidf = vectorizer.fit_transform([query, compressed])
    #         score = cosine_similarity(tfidf[0:1], tfidf[1:2])[0][0]
    #         if score > best_score:
    #             best_score = score
    #             best_text = compressed


In [ ]:
#@title 🎧 Listen: Todo1 After
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_14_todo1_after.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

In [ ]:
    return best_text, best_score

# Test your implementation
# result, score = find_optimal_compression(document, query, target_tokens=80)
# print(f"Best compression: {len(enc.encode(result))} tokens, relevance: {score:.3f}")

In [ ]:
#@title 🎧 Listen: Todo2 Intro
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_15_todo2_intro.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

**TODO Exercise 2:** Implement a dynamic budget allocator based on query complexity.

In [ ]:
#@title 🎧 Listen: Todo2 Task
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_16_todo2_task.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

In [ ]:
def classify_query_complexity(query):
    """
    TODO: Classify query as 'simple', 'moderate', or 'complex'.
    Heuristics:
    - Simple: < 10 words, no comparison words
    - Complex: > 20 words OR contains compare/analyze/evaluate
    - Moderate: everything else
    """
    # YOUR CODE HERE


In [ ]:
#@title 🎧 Listen: Todo2 After
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_17_todo2_after.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

In [ ]:
    pass

def dynamic_allocate(window_size, reserved, query):
    """
    TODO: Allocate tokens based on query complexity.
    Formula: T_i = (w_i * s_i) / sum(w_j * s_j) * available
    """
    # YOUR CODE HERE
    pass

In [ ]:
#@title 🎧 Listen: Pipeline Intro
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_18_pipeline_intro.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

## 6. Putting It All Together: Compression Pipeline

In [ ]:
#@title 🎧 Listen: Visualization
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_19_visualization.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

In [ ]:
import matplotlib.pyplot as plt

ratios = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]
token_counts = []
relevance_scores = []

vectorizer = TfidfVectorizer(stop_words='english')

for ratio in ratios:
    if ratio >= 1.0:
        comp = document
    else:
        comp = extractive_compress(document, query, keep_ratio=ratio)
    token_counts.append(len(enc.encode(comp)))
    tfidf = vectorizer.fit_transform([query, comp])
    score = cosine_similarity(tfidf[0:1], tfidf[1:2])[0][0]
    relevance_scores.append(score)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.bar(range(len(ratios)), token_counts, color='steelblue', alpha=0.8)
ax1.set_xticks(range(len(ratios)))
ax1.set_xticklabels([f'{r:.0%}' for r in ratios], rotation=45)
ax1.set_xlabel('Keep Ratio')
ax1.set_ylabel('Token Count')
ax1.set_title('Tokens After Compression')

ax2.plot(ratios, relevance_scores, 'o-', color='green', linewidth=2)
ax2.set_xlabel('Keep Ratio')
ax2.set_ylabel('Relevance Score')
ax2.set_title('Information Retention vs Compression')
ax2.set_ylim(0, 1)

plt.tight_layout()
plt.savefig('compression_tradeoffs.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
#@title 🎧 Listen: Final Output Intro
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_20_final_output_intro.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

## 7. Final Output: Budget-Aware Compressor

In [ ]:
#@title 🎧 Listen: Budget Aware Impl
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_21_budget_aware_impl.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

In [ ]:
class BudgetAwareCompressor:
    def __init__(self, window_size=128_000, reserved_output=35_000):
        self.window_size = window_size
        self.reserved = reserved_output
        self.available = window_size - reserved_output
        self.enc = tiktoken.encoding_for_model("gpt-4")

    def count_tokens(self, text):
        return len(self.enc.encode(text))

    def compress_to_fit(self, documents, query, system_prompt, history=""):
        system_tokens = self.count_tokens(system_prompt)
        history_tokens = self.count_tokens(history) if history else 0
        query_tokens = self.count_tokens(query)
        rag_budget = self.available - system_tokens - history_tokens - query_tokens

        compressed_docs = []
        used_tokens = 0

        for doc in documents:
            remaining = rag_budget - used_tokens
            if remaining <= 0:
                break
            doc_tokens = self.count_tokens(doc)
            if doc_tokens <= remaining:
                compressed_docs.append(doc)
                used_tokens += doc_tokens
            else:
                target_ratio = remaining / doc_tokens
                comp = extractive_compress(doc, query, keep_ratio=min(target_ratio, 0.5))
                comp_tokens = self.count_tokens(comp)
                if comp_tokens <= remaining:
                    compressed_docs.append(comp)
                    used_tokens += comp_tokens

        return {
            "documents": compressed_docs,
            "token_breakdown": {
                "system": system_tokens, "history": history_tokens,
                "rag": used_tokens, "query": query_tokens,
                "total": system_tokens + history_tokens + used_tokens + query_tokens,
                "budget": self.available
            }
        }

compressor = BudgetAwareCompressor()
result = compressor.compress_to_fit(
    documents=[document] * 3,
    query="How does RAG reduce hallucination?",
    system_prompt="You are a helpful AI assistant.",
    history="User: What is RAG?\nAssistant: RAG stands for Retrieval-Augmented Generation."
)

print("=== Budget-Aware Compression Result ===")
for k, v in result["token_breakdown"].items():
    print(f"  {k}: {v:,} tokens")
print(f"Documents included: {len(result['documents'])}")

In [ ]:
#@title 🎧 Listen: Closing
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_22_closing.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

## 8. Reflection and Next Steps

**Key takeaways:**
- Not all tokens are equal — prioritize by relevance to the query
- Compression ratios of 70-80% are achievable while retaining 90%+ of relevant information
- Always measure both compression AND information retention

**Next notebook:** RAGAS evaluation framework for measuring context quality.